In [1]:
from pathlib import Path
import os
from collections import defaultdict

from caf.base import DVector, ZoningSystem
from caf.base.zoning import TranslationWeighting
import pandas as pd

os.chdir(r'..\..')
from land_use.constants.geographies import CACHE_FOLDER, GORS, KNOWN_GEOGRAPHIES
from land_use.data_processing.mapping import create_interactive_maps
from land_use.data_processing import create_dvector_from_data, read_dvector_data, translate_and_combine_dvectors, generate_segment_bar_plots
os.chdir(r'supporting_processes\norcom')

In [2]:
# path to save any outputs
working_dir = Path(r"I:\NorMITs NorCOM\Validation\post-adjustment\v38\2023\DVLA")

In [3]:
# keep required types of car from DVLA for comparison with Census (exclude company cars for example)
# this is based on an email from Adil 15/04/2025
relevant_classifications = {
    "BodyType": ["Cars", "Other body types"],
    "Keepership": ["Private"],
    "LicenceStatus": ["Licensed", "SORN"]
}
# define columns to average number of cars over
data_cols = ["2023 Q4", "2023 Q3", "2023 Q2", "2023 Q1"]

In [4]:
# read in DVLA data and convert to DVector
dvla_data = Path(r"I:\NorMITs NorCOM\Import\DVLA\df_VEH0125 number of cars by quarter by LSOA11.csv")
dvla = pd.read_csv(dvla_data, low_memory=False)
output_dir = working_dir / "dvla_data"
output_dir.mkdir(exist_ok=True, parents=True)

In [5]:
# copy data to avoid keeping reading in
df = dvla.copy()

# filter down based on vehicle and ownership type classifications
for col, relevant_values in relevant_classifications.items():
    df = df[df[col].isin(relevant_values)]

# filter down to columns that are relevant for 2023
df = df[["LSOA11CD"] + data_cols]

# enforce columns to be numeric (low values are provided as [x] or [c]) and infill with 0
for col in data_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# sum the relevant cars
df = df.groupby("LSOA11CD", as_index=False)[data_cols].sum()

# average number of cars over the year
df["cars"] = df[data_cols].mean(axis=1)

In [6]:
# format for DVector
df["total"] = 1
df = df.pivot_table(values="cars", columns="LSOA11CD", index="total")
dvla_dvec = create_dvector_from_data(
    dvector_data=df, geographical_level="LSOA2011",
    input_segments=["total"]
)
dvla_dvec = dvla_dvec.translate_zoning(
    new_zoning=KNOWN_GEOGRAPHIES.get(f'LSOA2021'),
    cache_path=CACHE_FOLDER
)
dvla_dvec.save(output_dir / f"2023_dvla.hdf")

The input data started with 42,620 columns. Filtering to None results in 34,753 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\zoning.py:681: TranslationWarning: 304 LSOA2011 zones have splitting factors which don't sum to 1 (value totals may change during zone_translation), the maximum difference is 4.4e-16
  warnings.warn(
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")


In [7]:
# create GOR specific DVLA Dvectors
for GOR in GORS:
    # create a dvector
    regional = create_dvector_from_data(
        dvector_data=dvla_dvec.data, geographical_level="LSOA2021",
        input_segments=["total"], geography_subset=GOR
    )
    # save regional outputs
    regional.save(output_dir / f"2023_dvla_{GOR}.hdf")

The input data started with 35,672 columns. Filtering to NE results in 1,682 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data started with 35,672 columns. Filtering to NW results in 4,567 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data started with 35,672 columns. Filtering to YH results in 3,355 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data started with 35,672 columns. Filtering to Wales results in 1,917 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.p

In [8]:
# read in data about what the 2+ cars category should relate to by LSOA (produced by Adil, not sure how / where)
factors = pd.read_csv(r"I:\NorMITs NorCOM\Import\DVLA\FINAL_lsoa_factors_ons.csv").rename(columns={"Lower layer Super Output Areas Code": "LSOA2021"})
output_dir = working_dir / "2+factor_data"
output_dir.mkdir(exist_ok=True, parents=True)

# create car_availability dimension data
factors[1] = 0
factors[2] = 1
factors[3] = factors["factors"]
factors = factors[["LSOA2021", 1, 2, 3]].set_index("LSOA2021").T
factors.index.name = "car_availability"

In [9]:
# create GOR specific DVLA Dvectors
for GOR in GORS:
    try:
        # create a dvector
        regional = create_dvector_from_data(
            dvector_data=factors, geographical_level="LSOA2021",
            input_segments=["car_availability"], geography_subset=GOR
        )
        # save regional outputs
        regional.save(output_dir / f"lsoa_car_factors_{GOR}.hdf")
    except KeyError as exc:
        print(exc)
        print(f"*** Missing zones in data for {GOR} - SKIPPING ***")

The input data started with 33,754 columns. Filtering to NE results in 1,682 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data started with 33,754 columns. Filtering to NW results in 4,567 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data started with 33,754 columns. Filtering to YH results in 3,355 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data started with 33,754 columns. Filtering to WM results in 3,574 columns.


"None of [Index(['W01000003', 'W01000004', 'W01000005', 'W01000006', 'W01000007',\n       'W01000008', 'W01000009', 'W01000010', 'W01000011', 'W01000012',\n       ...\n       'W01002031', 'W01002032', 'W01002033', 'W01002034', 'W01002035',\n       'W01002036', 'W01002037', 'W01002038', 'W01002039', 'W01002040'],\n      dtype='object', name='LSOA2021', length=1917)] are in the [columns]"
*** Missing zones in data for Wales - SKIPPING ***


C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data started with 33,754 columns. Filtering to EM results in 2,847 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data started with 33,754 columns. Filtering to SW results in 3,407 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data started with 33,754 columns. Filtering to EoE results in 3,758 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum

"['E01034220'] not in index"
*** Missing zones in data for Lon - SKIPPING ***


C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")


In [10]:
# define path to NorCOM validation outputs
OUTPUT_DIR = Path(r"I:\NorMITs NorCOM\Validation\post-adjustment\v38\2023\data")
RESULTS_DIR = working_dir / "resulting_car_data"
RESULTS_DIR.mkdir(exist_ok=True)

In [11]:
# read in NorCOM outputs (households with number of cars) and convert to approx number of cars
for GOR in GORS:
    try:
        car_converter = DVector.load(working_dir / "2+factor_data" / f"lsoa_car_factors_{GOR}.hdf")
    except FileNotFoundError as exc:
        print(exc)
        print(f"*** No file of zonal factors for {GOR} - SKIPPING ***")
        continue
    modelled = DVector.load(OUTPUT_DIR / f'Expected_{GOR}.hdf')
    approx_cars = modelled * car_converter
    approx_cars = approx_cars.add_segments(['total']).aggregate(['total'])
    approx_cars.save(RESULTS_DIR / f'Approx_Cars_{GOR}.hdf')

File I:\NorMITs NorCOM\Validation\post-adjustment\v38\2023\DVLA\2+factor_data\lsoa_car_factors_Wales.hdf does not exist
*** No file of zonal factors for Wales - SKIPPING ***
File I:\NorMITs NorCOM\Validation\post-adjustment\v38\2023\DVLA\2+factor_data\lsoa_car_factors_Lon.hdf does not exist
*** No file of zonal factors for Lon - SKIPPING ***


In [12]:
# read in households with number of cars and dvla stuff and convert to differences
for GOR in GORS:
    try:
        approx_cars = DVector.load(working_dir / "resulting_car_data" / f'Approx_Cars_{GOR}.hdf')
        dvla = DVector.load(working_dir / "dvla_data" / f"2023_dvla_{GOR}.hdf")
    except FileNotFoundError as exc:
        print(exc)
        print(f"*** No file for {GOR} - SKIPPING ***")
        continue
    difference = approx_cars - dvla
    difference.save(RESULTS_DIR / f'diff_from_dvla_{GOR}.hdf')

File I:\NorMITs NorCOM\Validation\post-adjustment\v38\2023\DVLA\resulting_car_data\Approx_Cars_Wales.hdf does not exist
*** No file for Wales - SKIPPING ***
File I:\NorMITs NorCOM\Validation\post-adjustment\v38\2023\DVLA\resulting_car_data\Approx_Cars_Lon.hdf does not exist
*** No file for Lon - SKIPPING ***


In [13]:
# create lists of data to use
total_file_dict = defaultdict(list)
northern_file_dict = defaultdict(list)
northern_lsoa_dict = defaultdict(list)

dvla_path = working_dir / "dvla_data"
# get files from existing output
total_file_dict['DVLA'].extend(dvla_path.glob('2023_dvla_*.hdf'))
total_file_dict['Approximated'].extend(RESULTS_DIR.glob('Approx_Cars_*.hdf'))
northern_file_dict['DVLA'].extend([dvla_path / "2023_dvla_NW.hdf", dvla_path / "2023_dvla_NE.hdf", dvla_path / "2023_dvla_YH.hdf"])
northern_file_dict['Approximated'].extend([RESULTS_DIR / "Approx_Cars_NW.hdf", RESULTS_DIR / "Approx_Cars_NE.hdf", RESULTS_DIR / "Approx_Cars_YH.hdf"])        
northern_lsoa_dict['Difference'].extend([RESULTS_DIR / "diff_from_dvla_NW.hdf", RESULTS_DIR / "diff_from_dvla_NE.hdf", RESULTS_DIR / "diff_from_dvla_YH.hdf"])        

In [14]:
MAPPING_INPUTS = {
    'LSOA21-NORTH': northern_lsoa_dict,
    'LAD2021+SCOTLANDLAD': total_file_dict,
    'MSOA21-NORTH': northern_file_dict
}
for MAP_ZONE_SYSTEM, file_dict in MAPPING_INPUTS.items():
    # Calculate all of our "total" dictionaries in one go
    data_dict = {}
    for key, input_files in file_dict.items():
        print(key)
        if not input_files:
            continue
        
        data_dict[key] = translate_and_combine_dvectors(
            input_files=input_files,
            aggregate_zone_system=MAP_ZONE_SYSTEM
        )
    for unit, map_total_dvector in data_dict.items():
        # Set up the output directory for that unit category
        results_dir = RESULTS_DIR / MAP_ZONE_SYSTEM / unit
        results_dir.mkdir(exist_ok=True, parents=True)
    
        # Store all segment names so we can figure out which ones we skip
        all_segment_names = set(map_total_dvector.segmentation.names)
    
        for segment in all_segment_names:
    
            # Make the maps
            map_paths = create_interactive_maps(
                map_total_dvector, output_folder=results_dir, 
                specific_segment=segment,
                filter_by=None,
                simplification=100,
                fix_proportional_scale=True
            )

Difference
DVLA
Approximated
DVLA
Approximated


In [15]:
# Calculate all of our "total" dictionaries in one go
data_dict = {}
for key, input_files in total_file_dict.items():
    print(key)
    if not input_files:
        continue
    
    data_dict[key] = translate_and_combine_dvectors(
        input_files=input_files,
        aggregate_zone_system='LAD2021+SCOTLANDLAD'
    )
for unit, map_total_dvector in data_dict.items():
    # Set up the output directory for that unit category
    results_dir = RESULTS_DIR / 'LAD2021+SCOTLANDLAD' / unit
    results_dir.mkdir(exist_ok=True, parents=True)

    chart_total_dvector = map_total_dvector.translate_zoning(
        ZoningSystem.get_zoning('LAD2021+SCOTLANDLAD', search_dir=CACHE_FOLDER)
    )

    # Store all segment names so we can figure out which ones we skip
    all_segment_names = set(chart_total_dvector.segmentation.names)

    for segment_plot in generate_segment_bar_plots(chart_total_dvector, unit=unit):
        # And save the data
        segment_plot.summary_data.to_csv(
            results_dir / f'{segment_plot.segments}.csv', 
            float_format=lambda x: '{:,.0f}'.format(x)
        )
    

DVLA
Approximated
